In [16]:
import pandas as pd
#1 task
class TabularDataset:
    def __init__(self,a):
        self._df = pd.DataFrame(a)

    def data(self):
        return self._df.copy()

    def shape(self):
        return f"shape:{self._df.shape}"

    def column_names(self):
        return f"column names:{list(self._df.columns)}"

    def head_info(self, n: int = 5) -> str:
        return f"head {n}:{str(self._df.head(n))}"

    def numeric_summary(self) -> pd.DataFrame:
        return self._df.describe()

dc = TabularDataset({
    'name':['Danial','Aibol','Julia','Alice'],
    'age':[18,22,19,23],
    'score':[35,22,45,89]
})
dc.shape(),dc.head_info(1),dc.numeric_summary()

('shape:(4, 3)',
 'head 1:     name  age  score\n0  Danial   18     35',
              age     score
 count   4.000000   4.00000
 mean   20.500000  47.75000
 std     2.380476  29.06745
 min    18.000000  22.00000
 25%    18.750000  31.75000
 50%    20.500000  40.00000
 75%    22.250000  56.00000
 max    23.000000  89.00000)

In [30]:
#task 2
import pathlib
class CsvRepository:
    def __init__(self,path: str | pathlib.Path,required_columns: list[str]):
        self.path = path
        self.required_columns = required_columns

    def load(self) -> TabularDataset:
        dt = pd.read_csv(self.path)
        if not self.required_columns:
            raise ValueError("There must to be columns")
        
        return TabularDataset(dt)

    def save(self, datacet: TabularDataset)  -> None:
        dataset._df.to_csv(self.path,index=False)
        print(f"The file {self.path} has saved successfuly")

dt_repo = CsvRepository("plot_data.csv", ["time", "pressure"])
dataset = dt_repo.load()
print(dataset.shape())
new_data = dataset._df.copy()
new_data.iloc[0, 0] = 999  
new_dataset = TabularDataset(new_data)

dt_repo.save(new_dataset)

shape:(5, 4)
The file plot_data.csv has saved successfuly


In [31]:
#task 3
from abc import ABC, abstractmethod
class DataFrameTransform:
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class FilterByScoreTransform(DataFrameTransform):
    def __init__(self,min_score: float,column_name: str = "score"):
        self.min_score = min_score
        self.column_name = column_name

    def apply(self, df:pd.DataFrame) -> pd.DataFrame:
        new_df = df.copy()
        return new_df[new_df[self.column_name]>=self.min_score]

class AddPassedColumnTransform(DataFrameTransform):
    def apply(self, df:pd.DataFrame) -> pd.DataFrame:
        new_df = df.copy()
        new_df["passed"] = new_df["score"]>=70
        return new_df

data = pd.DataFrame({
    "name": ["Ivan", "Dania", "Oleg", "Anna"],
    "score": [85, 40, 75, 20]
})

filter_transform = FilterByScoreTransform(min_score=50)
filtered_data = filter_transform.apply(data)

passed_transform = AddPassedColumnTransform()
final_data = passed_transform.apply(filtered_data)

print(final_data)


   name  score  passed
0  Ivan     85    True
2  Oleg     75    True


In [33]:
#task 4
class PandasTransformPipeline:
    def __init__(self, *transforms: DataFrameTransform):
        self.transforms = transforms

    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        result_df = df.copy()
        
        for transform in self.transforms:
            result_df = transform.apply(result_df)
            
        return result_df

    def describe(self) -> str:
        names = [type(t).__name__ for t in self.transforms]
        return " -> ".join(names)

data = pd.DataFrame({
    "name": ["Ivan", "Dania", "Oleg", "Anna", "Masha", "Alex"],
    "score": [85, 40, 75, 20, 95, 60]
})


pipeline = PandasTransformPipeline(
    FilterByScoreTransform(min_score=50), 
    AddPassedColumnTransform()            
)

print(f"Pipeline steps: {pipeline.describe()}")


final_df = pipeline.run(data)


print(final_df)


Pipeline steps: FilterByScoreTransform -> AddPassedColumnTransform
    name  score  passed
0   Ivan     85    True
2   Oleg     75    True
4  Masha     95    True
5   Alex     60   False


In [34]:
#task 5
import pandas as pd

class GroupScoreReport:
    def __init__(self, df: pd.DataFrame, student_col="student", subject_col="subject", score_col="score"):
        self._df = df.copy()
        self.student_col = student_col
        self.subject_col = subject_col
        self.score_col = score_col

    def aggregate_by_student(self) -> pd.DataFrame:
        report = self._df.groupby(self.student_col)[self.score_col].agg(
            count='count',
            mean='mean',
            min='min',
            max='max'
        )
        return report

    def pivot_subjects(self) -> pd.DataFrame:
        pivot = self._df.pivot_table(
            index=self.student_col,
            columns=self.subject_col,
            values=self.score_col,
            aggfunc='mean'
        )
        return pivot

data = pd.DataFrame({
    "student": ["Ivan", "Ivan", "Dania", "Dania", "Oleg", "Oleg", "Anna", "Anna"],
    "subject": ["Math", "Physics", "Math", "Physics", "Math", "Physics", "Math", "Physics"],
    "score": [80, 90, 70, 60, 100, 95, 85, 80]
})


report_tool = GroupScoreReport(data)

print(report_tool.aggregate_by_student())


print(report_tool.pivot_subjects())


         count  mean  min  max
student                       
Anna         2  82.5   80   85
Dania        2  65.0   60   70
Ivan         2  85.0   80   90
Oleg         2  97.5   95  100
subject   Math  Physics
student                
Anna      85.0     80.0
Dania     70.0     60.0
Ivan      80.0     90.0
Oleg     100.0     95.0


In [37]:
#task 6
import pandas as pd

class TableLinker:

    def merge_left(self, left: pd.DataFrame, right: pd.DataFrame, on: str) -> pd.DataFrame:

        return pd.merge(left, right, on=on, how="left")

    def concat_rows(self, parts: list[pd.DataFrame]) -> pd.DataFrame:
        return pd.concat(parts, ignore_index=True, axis=0)

linker = TableLinker()

df_left = pd.DataFrame({"id": [1, 2, 3], "name": ["Alice", "Bob", "Charlie"]})
df_right = pd.DataFrame({"id": [1, 2, 4], "city": ["NY", "LA", "Tokyo"]})

merged = linker.merge_left(df_left, df_right, on="id")
print(merged)


chunk1 = pd.DataFrame({"name": ["Ivan"], "score": [85]})
chunk2 = pd.DataFrame({"name": ["Maria"], "score": [92]})

stacked = linker.concat_rows([chunk1, chunk2])
print(stacked)


   id     name city
0   1    Alice   NY
1   2      Bob   LA
2   3  Charlie  NaN
    name  score
0   Ivan     85
1  Maria     92


In [38]:
#task 7
from abc import ABC, abstractmethod
import pandas as pd

class MissingValueStrategy(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass


class FillNumericMeanStrategy(MissingValueStrategy):
    def __init__(self, columns: list):
        self.columns = columns

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        new_df = df.copy()
        for col in self.columns:
            if not pd.api.types.is_numeric_dtype(new_df[col]):
                raise ValueError(f"Колонка {col} не является числовой!")
            
            mean_val = new_df[col].mean()
            new_df[col] = new_df[col].fillna(mean_val)
        return new_df


class DropRowsAnyStrategy(MissingValueStrategy):
    def __init__(self, columns: list = None):
        self.columns = columns 

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna(subset=self.columns)

class MissingValueProcessor:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def process(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.apply(df)

data = pd.DataFrame({
    "name": ["Ivan", "Dania", "Oleg", "Anna", "Masha"],
    "score": [85, None, 70, None, 90]
})

print(f"Размер ДО: {data.shape}")

filler = MissingValueProcessor(FillNumericMeanStrategy(columns=["score"]))
df_filled = filler.process(data)
print(f"Размер после Fill: {df_filled.shape}")
print(df_filled)

dropper = MissingValueProcessor(DropRowsAnyStrategy(columns=["score"]))
df_dropped = dropper.process(data)
print(f"\nРазмер после Drop: {df_dropped.shape}")
print(df_dropped)


Размер ДО: (5, 2)
Размер после Fill: (5, 2)
    name      score
0   Ivan  85.000000
1  Dania  81.666667
2   Oleg  70.000000
3   Anna  81.666667
4  Masha  90.000000

Размер после Drop: (3, 2)
    name  score
0   Ivan   85.0
2   Oleg   70.0
4  Masha   90.0


In [41]:
def main():
    print("Ayapov Danial")
    print("CSS-1004-6-Ch")


    raw_data = pd.DataFrame({
        "student": ["Ivan", "Dania", "Oleg", "Anna", "Masha", "Ivan", "Dania"],
        "subject": ["Math", "Math", "Math", "Math", "Math", "Physics", "Physics"],
        "score": [85, None, 45, 90, 30, 75, None]
    })

    cleaner = MissingValueProcessor(FillNumericMeanStrategy(columns=["score"]))
    data_cleaned = cleaner.process(raw_data)


    pipeline = PandasTransformPipeline(
        FilterByScoreTransform(min_score=40),
        AddPassedColumnTransform()
    )
    data_transformed = pipeline.run(data_cleaned)


    report = GroupScoreReport(data_transformed)
    agg_report = report.aggregate_by_student()
    print(agg_report)


    groups_df = pd.DataFrame({
        "student": ["Ivan", "Dania", "Oleg", "Anna", "Masha"],
        "group_id": ["A1", "A1", "B2", "B2", "A1"]
    })
    
    linker = TableLinker()

    final_table = linker.merge_left(data_transformed, groups_df, on="student")
    
    print(final_table.head())

if __name__ == "__main__":
    main()


Ayapov Danial
CSS-1004-6-Ch
         count  mean   min   max
student                         
Anna         1  90.0  90.0  90.0
Dania        2  65.0  65.0  65.0
Ivan         2  80.0  75.0  85.0
Oleg         1  45.0  45.0  45.0
  student  subject  score  passed group_id
0    Ivan     Math   85.0    True       A1
1   Dania     Math   65.0   False       A1
2    Oleg     Math   45.0   False       B2
3    Anna     Math   90.0    True       B2
4    Ivan  Physics   75.0    True       A1
